# Sentence-End Detection — Feature Analysis & Model Training

Сравнение MFA-признаков (точных фонетических) с акустическими.
Обучение на GPU: LR, XGBoost, MLP.

In [ ]:
!pip install -q xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report, RocCurveDisplay
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                       capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT AVAILABLE')

## 1. Загрузка данных

In [ ]:
import os

# Загрузка CSV из репо (уже в папке)
CSV_PATH = 'sentence_end_features.csv'

df = pd.read_csv(CSV_PATH)
print(f'Rows: {len(df):,}  |  sentence-ends: {df["y"].sum():,} ({df["y"].mean()*100:.1f}%)')
print(f'Sources: {df["source"].unique().tolist()}')
df.head()

## 2. Per-feature AUC (что реально работает)

In [ ]:
feature_cols = [
    'pause_sec', 'log_pause',
    'last_phone_dur', 'phone_dur_ratio',
    'last_vowel_dur', 'vowel_ratio',
    'word_dur', 'word_dur_ratio',
    'n_phones', 'phone_variability',
    'rate_before', 'rate_after', 'rate_delta',
]

rows = []
for c in feature_cols:
    vals = df[c].fillna(0)
    auc = roc_auc_score(df['y'], vals)
    if auc < 0.5:
        auc = 1 - auc
    rows.append({'feature': c,
                 'auc': auc,
                 'neg_mean': df[df['y']==0][c].mean(),
                 'pos_mean': df[df['y']==1][c].mean()})

feat_df = pd.DataFrame(rows).sort_values('auc', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(feat_df['feature'], feat_df['auc'] - 0.5, left=0.5)
ax.axvline(0.5, color='black', linewidth=0.8)
ax.set_xlabel('AUC (single feature)')
ax.set_title('Feature Separability')
for bar, val in zip(bars, feat_df['auc']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()
feat_df

## 3. Модели: LR vs XGBoost (GPU) vs MLP

In [ ]:
X = df[feature_cols].fillna(0).values
y = df['y'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

results = {}

# --- Logistic Regression ---
lr = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
lr.fit(X_tr_s, y_tr)
lr_prob = lr.predict_proba(X_te_s)[:, 1]
results['LR (MFA)'] = roc_auc_score(y_te, lr_prob)
print(f'LR AUC:      {results["LR (MFA)"]:.4f}')

In [ ]:
# --- XGBoost (GPU) ---
try:
    device = 'cuda' if result.returncode == 0 and result.stdout.strip() else 'cpu'
except:
    device = 'cpu'

pos_w = (y_tr == 0).sum() / (y_tr == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_w,
    tree_method='hist',
    device=device,
    eval_metric='auc',
    random_state=42,
    verbosity=0,
)
xgb_model.fit(X_tr, y_tr,
              eval_set=[(X_te, y_te)],
              verbose=False)
xgb_prob = xgb_model.predict_proba(X_te)[:, 1]
results['XGBoost (GPU)'] = roc_auc_score(y_te, xgb_prob)
print(f'XGBoost AUC: {results["XGBoost (GPU)"]:.4f}  (device={device})')

In [ ]:
# --- PyTorch MLP (GPU) ---
import torch
import torch.nn as nn

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {dev}')

class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),  nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1),
        )
    def forward(self, x): return self.net(x).squeeze(1)

Xtr_t = torch.tensor(X_tr_s, dtype=torch.float32).to(dev)
ytr_t = torch.tensor(y_tr,   dtype=torch.float32).to(dev)
Xte_t = torch.tensor(X_te_s, dtype=torch.float32).to(dev)

pos_weight = torch.tensor([(y_tr==0).sum() / (y_tr==1).sum()]).to(dev)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

model = MLP(X_tr_s.shape[1]).to(dev)
opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=200)

dataset = torch.utils.data.TensorDataset(Xtr_t, ytr_t)
loader  = torch.utils.data.DataLoader(dataset, batch_size=512, shuffle=True)

best_auc = 0
for epoch in range(200):
    model.train()
    for xb, yb in loader:
        opt.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        opt.step()
    sched.step()
    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            prob = torch.sigmoid(model(Xte_t)).cpu().numpy()
        auc = roc_auc_score(y_te, prob)
        best_auc = max(best_auc, auc)
        print(f'  epoch {epoch+1:3d}: AUC={auc:.4f}')

results['MLP (GPU)'] = best_auc
print(f'\nMLP best AUC: {best_auc:.4f}')

## 4. Сравнение моделей

In [ ]:
# Acoustic-only baseline (Swift model features)
acoustic_cols = ['pause_sec', 'log_pause', 'rate_before', 'rate_after', 'rate_delta']
Xa = df[acoustic_cols].fillna(0).values
Xa_tr, Xa_te = train_test_split(Xa, test_size=0.2, random_state=42)[0], \
               train_test_split(Xa, test_size=0.2, random_state=42)[1]
Xa_tr, Xa_te = StandardScaler().fit(Xa[train_test_split(range(len(Xa)), test_size=0.2, random_state=42)[0]]) \
                               .transform(Xa_tr), \
               StandardScaler().fit(Xa[train_test_split(range(len(Xa)), test_size=0.2, random_state=42)[0]]) \
                               .transform(Xa_te)

# Fix: proper split
idx_tr, idx_te = train_test_split(range(len(df)), test_size=0.2, random_state=42, stratify=y)
sc2 = StandardScaler().fit(Xa[idx_tr])
lr_ac = LogisticRegression(max_iter=1000, class_weight='balanced').fit(sc2.transform(Xa[idx_tr]), y[idx_tr])
results['LR (acoustic-only)'] = roc_auc_score(y[idx_te], lr_ac.predict_proba(sc2.transform(Xa[idx_te]))[:,1])

print('=== AUC Summary ===')
for name, auc in sorted(results.items(), key=lambda x: -x[1]):
    bar = '█' * int((auc - 0.5) * 100)
    print(f'{name:<22} {auc:.4f}  {bar}')

fig, ax = plt.subplots(figsize=(8, 4))
names = list(results.keys())
aucs  = [results[n] for n in names]
colors = ['#2196F3' if 'acoustic' in n else '#4CAF50' if 'XGB' in n else '#FF9800' if 'MLP' in n else '#9C27B0'
          for n in names]
ax.barh(names, aucs, color=colors)
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='random')
ax.set_xlim(0.4, 1.0)
ax.set_xlabel('AUC-ROC')
ax.set_title('Model Comparison')
for i, v in enumerate(aucs):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center')
plt.tight_layout()
plt.show()

## 5. XGBoost Feature Importance

In [ ]:
imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

# LR coefs
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'lr_coef': lr.coef_[0]
}).sort_values('lr_coef', key=abs, ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(imp['feature'], imp['importance'])
ax1.set_title('XGBoost Feature Importance (gain)')
ax1.set_xlabel('Importance')

colors2 = ['#4CAF50' if c > 0 else '#F44336' for c in coef_df['lr_coef']]
ax2.barh(coef_df['feature'], coef_df['lr_coef'], color=colors2)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title('LR Coefficients (standardized)')
ax2.set_xlabel('Coefficient')

plt.tight_layout()
plt.show()

print('Top XGBoost features:')
print(imp.to_string(index=False))
print('\nLR coefficients:')
print(coef_df.to_string(index=False))

## 6. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 6))

probs_map = {
    'LR (MFA)': lr_prob,
    'XGBoost':  xgb_prob,
}
if 'MLP (GPU)' in results:
    model.eval()
    with torch.no_grad():
        mlp_prob = torch.sigmoid(model(Xte_t)).cpu().numpy()
    probs_map['MLP'] = mlp_prob

for name, prob in probs_map.items():
    fpr, tpr, _ = roc_curve(y_te, prob)
    auc = roc_auc_score(y_te, prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--',alpha=0.3)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Экспорт весов LR для Swift

In [ ]:
import json

swift_model = {
    'coef': lr.coef_[0].tolist(),
    'intercept': float(lr.intercept_[0]),
    'scaler': {
        'mean':  sc.mean_.tolist(),
        'scale': sc.scale_.tolist(),
    },
    'features': feature_cols,
    'auc_roc': float(results['LR (MFA)']),
}

with open('sentence_end_model_colab.json', 'w') as f:
    json.dump(swift_model, f, indent=2)

print('Saved: sentence_end_model_colab.json')
print(f'AUC:  {swift_model["auc_roc"]:.4f}')
print(f'Coef: {[round(c,4) for c in swift_model["coef"]]}')

# Download
try:
    from google.colab import files
    files.download('sentence_end_model_colab.json')
except:
    pass